# Paper Digest — Kaggle ML for Keywording and Query Intent\n\nExploratory baseline for the selected paper corpus. The R2V source catalog currently has 8 records; this notebook does not assume 10 PDFs are available. It must not report model performance until real source text and reviewed labels exist.\n\nGoals: (1) candidate keywords grounded in paper chunks, (2) classify research-question intent using human-reviewed labels, (3) export results with provenance.\n

In [ ]:
from pathlib import Path\nimport json, platform, sys\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import classification_report, ConfusionMatrixDisplay\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nimport matplotlib.pyplot as plt\n\nprint('Python:', sys.version)\nprint('Platform:', platform.platform())\nprint('pandas:', pd.__version__, 'NumPy:', np.__version__)\n

## 1. Load the real chunk corpus\n\nProvide `chunks.jsonl` from the PDF ingestion step as a Kaggle input dataset. Required fields: `paper_id`, `chunk_id`, `page_start`, `text`. No placeholder research text is generated.\n

In [ ]:
CHUNKS_PATH = Path('/kaggle/input/paper-digest-corpus/chunks.jsonl')\nOUT_DIR = Path('/kaggle/working/paper-digest-ml')\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nif CHUNKS_PATH.exists():\n    chunks = pd.read_json(CHUNKS_PATH, lines=True)\n    required = {'paper_id', 'chunk_id', 'page_start', 'text'}\n    missing = required - set(chunks.columns)\n    if missing: raise ValueError(f'Missing required columns: {sorted(missing)}')\n    chunks = chunks.dropna(subset=['text']).copy()\n    chunks['text'] = chunks['text'].astype(str)\n    print('Chunks:', len(chunks), 'Papers:', chunks['paper_id'].nunique())\nelse:\n    chunks = pd.DataFrame(columns=['paper_id','chunk_id','page_start','text'])\n    print('Corpus input not mounted. Attach the real chunks.jsonl dataset before running ML.')\n

## 2. Keywording baseline: TF-IDF\n\nExtract terms by chunk and retain source identifiers. This is a lexical baseline, not a semantic understanding model. Review candidate terms manually before adding them to the graph.\n

In [ ]:
def extract_keyword_candidates(frame: pd.DataFrame, top_k: int = 12) -> pd.DataFrame:\n    if frame.empty: return pd.DataFrame(columns=['paper_id','chunk_id','page_start','keyword','score','method','review_status'])\n    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_df=0.95, min_df=1, token_pattern=r'(?u)\\b[a-zA-Z][a-zA-Z0-9_-]{2,}\\b')\n    matrix = vectorizer.fit_transform(frame['text'].tolist())\n    terms = np.asarray(vectorizer.get_feature_names_out())\n    rows = []\n    for row_idx, row in enumerate(frame.itertuples(index=False)):\n        scores = matrix[row_idx].toarray().ravel()\n        if not scores.any(): continue\n        for term_idx in np.argsort(scores)[-top_k:][::-1]:\n            if scores[term_idx] <= 0: continue\n            rows.append({'paper_id': row.paper_id, 'chunk_id': row.chunk_id, 'page_start': row.page_start, 'keyword': terms[term_idx], 'score': float(scores[term_idx]), 'method': 'tfidf_chunk_baseline', 'review_status': 'unreviewed'})\n    return pd.DataFrame(rows)\n\nkeyword_candidates = extract_keyword_candidates(chunks)\nkeyword_candidates.to_csv(OUT_DIR / 'keyword_candidates.csv', index=False)\nkeyword_candidates.head(30)\n

## 3. Query-intent dataset\n\nSupply a manually reviewed CSV with `query_text` and `intent_label`. Optional fields: `query_id`, `split_group`, `review_status`. Keep synthetic test examples clearly labeled and do not present them as real user behavior.\n\nAllowed initial labels: `summarize`, `compare`, `find_method`, `find_dataset`, `find_metric`, `find_limitations`, `find_evidence`, `identify_gap`, `other_or_unclear`.\n

In [ ]:
INTENT_PATH = Path('/kaggle/input/paper-digest-intent-labels/intent_examples.csv')\nif INTENT_PATH.exists():\n    intents = pd.read_csv(INTENT_PATH)\n    required = {'query_text', 'intent_label'}\n    missing = required - set(intents.columns)\n    if missing: raise ValueError(f'Missing required columns: {sorted(missing)}')\n    if 'review_status' in intents.columns:\n        intents = intents[intents['review_status'].eq('reviewed')].copy()\n    print('Reviewed examples:', len(intents), 'Classes:', intents['intent_label'].nunique())\nelse:\n    intents = pd.DataFrame(columns=['query_text','intent_label'])\n    print('No reviewed intent labels mounted. Classifier training is skipped.')\n

## 4. Train a small baseline only when data supports it\n\nA ten-paper corpus is not a substitute for a labeled query dataset. This cell requires at least two examples per class and enough samples for a stratified holdout; otherwise it deliberately skips training. For tiny datasets, use rule-based routing and collect labels first.\n

In [ ]:
intent_model = None\nif len(intents) >= 20 and intents['intent_label'].value_counts().min() >= 2:\n    X_train, X_test, y_train, y_test = train_test_split(\n        intents['query_text'], intents['intent_label'], test_size=0.25, random_state=42, stratify=intents['intent_label']\n    )\n    intent_model = Pipeline([\n        ('tfidf', TfidfVectorizer(ngram_range=(1,2), lowercase=True, sublinear_tf=True)),\n        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))\n    ])\n    intent_model.fit(X_train, y_train)\n    predictions = intent_model.predict(X_test)\n    print(classification_report(y_test, predictions, zero_division=0))\nelse:\n    print('Skipped: collect more human-reviewed intent labels before training or reporting metrics.')\n\nif intent_model is not None:\n    pd.DataFrame({'query_text': X_test, 'actual_intent': y_test, 'predicted_intent': predictions}).to_csv(OUT_DIR / 'intent_predictions.csv', index=False)\n

## 5. Review and export\n\nDo not feed unreviewed keywords directly into the canonical graph. Review precision@k on a manually sampled set, audit intent confusion, and preserve the source chunk/page for every accepted keyword.\n\nExports: `keyword_candidates.csv` and, when training runs, `intent_predictions.csv`. This notebook has not been executed against the user's PDFs; there are no measured scores yet.\n